# Hodge O(u^4) full-T1 occurrence preflight

Run the single code cell below. It uploads and hash-checks the supplied Python file, mounts Google Drive, creates a new timestamped run directory, performs the bounded 63-unit CPU preflight, and independently verifies the finished certificate and checkpoint. A prior checkpoint is never reused by this cell. A resumed command-line run is diagnostic only and cannot produce a promotable PASS. The portable certificate ID contains no raw floats or checkpoint integrity hashes; the full checkpoint chain and file hashes remain separately verified external integrity metadata. The artifact stops at the occurrence boundary and never authorizes a GPU or coefficient run. Default ceilings are 6 hours, 48 GiB RAM, 50 million pair tests, and a 64 MiB JSON payload.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import math
import os
import subprocess
import sys
from google.colab import drive, files

EXPECTED_SCRIPT_SHA256 = 'A4539D603248BEA8589246F3CA982D8901F0AB826BD8849BD9DB7BAEF0BAC190'
EXPECTED_AUTHORITY_SHA256 = '935A3A5BA680D1373A5842486B10231D83232D8CB3393BBC250351BC51A68C8B'
EXPECTED_GATE_NAMES = (
    'negative occurrence poison callback remains untouched',
    'negative schedule poison callback remains untouched',
    'contractor domain is bound to the executed v10a2 reference manifest',
    'all Haar diagnostics are finite',
    'balanced rank-three inverse certificate',
    'pure-six singlet rank certificate',
    'pure-six projector trace equals five',
    'pure-six projector is symmetric',
    'pure-six projector is idempotent',
    'all three full-T1 root histories are constructed',
    'each root history has exactly two sealed magnetic applications',
    'all magnetic transition statistics satisfy operational bounds',
    'all reduced resolvents are finite and free of retained-energy poles',
    'all 63 full-T1 census units are complete and substantive where required',
    'every left H0 block scans exactly 125 translations',
    'all ordered 3x3 polarization pairs are covered for every moment',
    'D same-polarization analytic and all cross-polarization generic routes are exact',
    'observed center-neutral occurrences are contained in the executed corpus',
    'forbidden seven-factor occurrences are absent',
    'all local occurrences have at most six factors',
    'checkpoint binding and unit completeness are exact',
    'policy: magnetic block ledger is Hermitian-closed',
    'policy: static prohibited-token scope scan is clear',
)
EXPECTED_MOMENTS = ('e1', 'K2/e2', 'sigma3', 'N', 'C1', 'J', 'D')
EXPECTED_UNITS = tuple(
    f'{moment}:bra{bra}->ket{ket}'
    for moment in EXPECTED_MOMENTS for ket in range(3) for bra in range(3)
)
EXPECTED_NONVACUITY_MATRIX = {
    'e1': [[True, False, False], [False, True, False], [False, False, True]],
    'K2/e2': [[True] * 3 for _ in range(3)],
    'sigma3': [[False] * 3 for _ in range(3)],
    'N': [[True] * 3 for _ in range(3)],
    'C1': [[True] * 3 for _ in range(3)],
    'J': [[True] * 3 for _ in range(3)],
    'D': [[True] * 3 for _ in range(3)],
}

def require(condition, message):
    if not condition:
        raise RuntimeError(message)

def assert_finite(value, label='$'):
    if isinstance(value, float):
        require(math.isfinite(value), f'Non-finite number at {label}: {value!r}')
    elif isinstance(value, dict):
        require(all(isinstance(key, str) for key in value), f'Non-string JSON key at {label}')
        for key, item in value.items():
            assert_finite(item, f'{label}.{key}')
    elif isinstance(value, list):
        for index, item in enumerate(value):
            assert_finite(item, f'{label}[{index}]')
    elif not isinstance(value, (str, int, bool, type(None))):
        raise RuntimeError(f'Unexpected JSON value at {label}: {type(value).__name__}')

def reject_constant(token):
    raise ValueError(f'Non-finite JSON constant: {token}')

def reject_duplicate_keys(pairs):
    value = {}
    for key, item in pairs:
        require(key not in value, f'Duplicate JSON key: {key!r}')
        value[key] = item
    return value

def strict_load(path):
    value = json.loads(
        path.read_text(encoding='utf-8'),
        parse_constant=reject_constant, object_pairs_hook=reject_duplicate_keys,
    )
    assert_finite(value, str(path))
    return value

def canonical_json(value):
    assert_finite(value)
    return json.dumps(
        value, sort_keys=True, separators=(',', ':'), ensure_ascii=True, allow_nan=False,
    )

def value_sha256(value):
    return hashlib.sha256(canonical_json(value).encode('utf-8')).hexdigest().upper()

def assert_portable_identity(value, label='$'):
    forbidden = {
        'checkpoint_integrity', 'chain_head_sha256', 'file_sha256',
        'record_sha256', 'previous_record_sha256',
    }
    if isinstance(value, float):
        raise RuntimeError(f'Portable certificate identity contains raw float at {label}')
    if isinstance(value, dict):
        for key, item in value.items():
            require(key not in forbidden, f'Portable identity contains external checkpoint field {key!r} at {label}')
            assert_portable_identity(item, f'{label}.{key}')
    elif isinstance(value, list):
        for index, item in enumerate(value):
            assert_portable_identity(item, f'{label}[{index}]')

def file_sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest().upper()

os.chdir('/content')
script = Path('/content/ENGINE_O4_hodge_rootface_occurrence_preflight_colab.py')
if not script.is_file():
    uploaded = files.upload()
    require(script.name in uploaded, f'Upload the exact {script.name} supplied with this notebook.')
require(script.is_file(), f'Missing uploaded script: {script}')
require(file_sha256(script) == EXPECTED_SCRIPT_SHA256, 'Uploaded script SHA-256 mismatch.')

drive.mount('/content/drive')
stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
run_dir = (
    Path('/content/drive/MyDrive/Hodge_O4_FullT1_Preflight')
    / f'fresh_{stamp}_{EXPECTED_SCRIPT_SHA256[:12]}'
)
run_dir.mkdir(parents=True, exist_ok=False)
certificate = run_dir / 'CERT_O4_hodge_fullt1_occurrence_preflight.json'
checkpoint = run_dir / 'Hodge_O4_FullT1_Occurrence_Preflight.checkpoint.json'
command = [
    sys.executable, str(script),
    '--json-out', str(certificate),
    '--checkpoint', str(checkpoint),
    '--notebook-expected-script-sha', EXPECTED_SCRIPT_SHA256,
    '--max-payload-mib', '64',
]
completed = subprocess.run(command, check=False)
if completed.returncode != 0:
    failure = strict_load(certificate) if certificate.is_file() else {}
    raise RuntimeError(
        f'Preflight failed with exit {completed.returncode}: '
        f'{failure.get("error_type")}: {failure.get("error")}'
    )

result = strict_load(certificate)
require(result.get('schema') == 'hodge-o4-full-t1-occurrence-preflight/v4', 'Certificate schema mismatch.')
require(result.get('status') == 'PASS', f'Fresh run did not PASS: {result.get("status")}')
require(result.get('promotable_pass') is True, 'Fresh result is not promotable.')

runtime = result.get('runtime_provenance')
require(isinstance(runtime, dict), 'Missing runtime provenance.')
require(file_sha256(script) == runtime.get('script_sha256') == EXPECTED_SCRIPT_SHA256, 'Runtime script hash mismatch.')
require(runtime.get('notebook_expected_script_sha256') == EXPECTED_SCRIPT_SHA256, 'Notebook/script binding mismatch.')
require(runtime.get('authority_sha256') == EXPECTED_AUTHORITY_SHA256, 'Runtime authority hash mismatch.')
require(result.get('source_locators', {}).get('authority_sha256') == EXPECTED_AUTHORITY_SHA256, 'Source authority hash mismatch.')
authority_path = script.parent / str(runtime.get('authority_path'))
authority_status = runtime.get('authority_runtime_status')
if authority_status == 'verified':
    require(authority_path.is_file(), 'Authority was marked verified but is absent.')
    require(file_sha256(authority_path) == EXPECTED_AUTHORITY_SHA256, 'Uploaded authority hash mismatch.')
elif authority_status == 'not-present-in-uploaded-Colab-bundle':
    require(not authority_path.is_file(), 'Authority status says absent but a file is present.')
else:
    raise RuntimeError(f'Unknown authority runtime status: {authority_status!r}')
require(value_sha256(runtime.get('environment')) == runtime.get('environment_sha256'), 'Environment fingerprint hash mismatch.')
require(value_sha256(runtime.get('config')) == runtime.get('config_sha256'), 'Configuration hash mismatch.')
require(
    value_sha256(result.get('executed_v10a2_reference'))
    == result.get('executed_v10a2_reference_sha256'),
    'Executed reference hash mismatch.',
)
scope = result.get('certificate_identity_scope', {})
require(scope.get('environment_fingerprint_bound') is True, 'Environment fingerprint is not bound.')
require(scope.get('exact_environment_attestation') is False, 'Certificate overclaims exact environment attestation.')
require('not a container' in runtime.get('environment_binding_semantics', ''), 'Environment semantics are not disclosed.')

execution = result.get('execution', {})
require(execution.get('mode') == 'FRESH', 'Execution mode is not fresh.')
require(execution.get('resume_requested') is False, 'A resumed run cannot be promoted.')
require(execution.get('checkpoint_existed_at_open') is False, 'Fresh checkpoint existed before the run.')
require(execution.get('resumed_units') == [], 'Fresh certificate contains resumed units.')
require(execution.get('fresh_units') == list(EXPECTED_UNITS), 'Fresh unit ledger is not the exact 63-unit manifest.')
require(execution.get('promotable') is True, 'Execution disclosure is not promotable.')
require(len(EXPECTED_UNITS) == len(set(EXPECTED_UNITS)) == 63, 'Notebook unit manifest is malformed.')

gate_names = tuple(row.get('name') for row in result.get('gates', []))
require(len(EXPECTED_GATE_NAMES) == len(set(EXPECTED_GATE_NAMES)) == 23, 'Notebook gate manifest is malformed.')
require(gate_names == EXPECTED_GATE_NAMES, f'Gate manifest mismatch: {gate_names!r}')

identity = result.get('certificate_identity_material')
require(isinstance(identity, dict), 'Missing certificate identity material.')
assert_portable_identity(identity)
recomputed_certificate_id = value_sha256(identity)
require(result.get('certificate_id') == recomputed_certificate_id, 'Certificate ID does not match its identity material.')
require(scope.get('certificate_id') == recomputed_certificate_id, 'Certificate identity scope ID mismatch.')
require(identity.get('gate_names') == list(EXPECTED_GATE_NAMES), 'Identity material gate list mismatch.')
require(identity.get('execution') == {
    'mode': 'FRESH', 'resumed_units': [], 'fresh_units': list(EXPECTED_UNITS),
}, 'Identity material freshness mismatch.')

checkpoint_doc = strict_load(checkpoint)
require(checkpoint_doc.get('schema') == 'hodge-full-t1-occurrence-checkpoint/v2', 'Checkpoint schema mismatch.')
require(checkpoint_doc.get('unit_order') == list(EXPECTED_UNITS), 'Checkpoint unit order mismatch.')
require(set(checkpoint_doc.get('units', {})) == set(EXPECTED_UNITS), 'Checkpoint unit key mismatch.')
checkpoint_binding = checkpoint_doc.get('binding')
expected_binding = {
    'script_sha256': EXPECTED_SCRIPT_SHA256,
    'config_sha256': runtime.get('config_sha256'),
    'authority_sha256': EXPECTED_AUTHORITY_SHA256,
    'authority_runtime_status': authority_status,
    'environment_sha256': runtime.get('environment_sha256'),
    'executed_v10a2_reference_sha256': result.get('executed_v10a2_reference_sha256'),
}
require(checkpoint_binding == expected_binding, 'Checkpoint binding mismatch.')
require(result.get('checkpoint', {}).get('binding') == expected_binding, 'Certificate/checkpoint binding mismatch.')
require(identity.get('binding') == expected_binding, 'Identity/checkpoint binding mismatch.')

previous = None
for unit in EXPECTED_UNITS:
    record = checkpoint_doc['units'].get(unit)
    require(isinstance(record, dict) and record.get('unit') == unit, f'Missing checkpoint record: {unit}')
    require(record.get('previous_record_sha256') == previous, f'Checkpoint predecessor mismatch: {unit}')
    body = {
        'previous_record_sha256': record.get('previous_record_sha256'),
        'unit': record.get('unit'),
        'stats': record.get('stats'),
        'audit': record.get('audit'),
    }
    previous = value_sha256(body)
    require(record.get('record_sha256') == previous, f'Checkpoint record hash mismatch: {unit}')
    moment = unit.split(':', 1)[0]
    require(result.get('moments', {}).get(moment, {}).get(unit) == record.get('stats'), f'Certificate/checkpoint stats mismatch: {unit}')
    unit_audit = record.get('audit', {})
    occurrence_counts = unit_audit.get('all_occurrences', {})
    representatives = unit_audit.get('representatives', {})
    require(set(representatives) == set(occurrence_counts), f'Incomplete representative ledger: {unit}')
    require(result.get('occurrence_audit', {}).get('by_consumer', {}).get(unit, {}) == occurrence_counts, f'Occurrence ledger mismatch: {unit}')
    require(result.get('occurrence_audit', {}).get('pair_tests', {}).get(unit, 0) == unit_audit.get('pair_tests'), f'Pair-test ledger mismatch: {unit}')
    for pattern, representative in representatives.items():
        require(representative.get('consumer') == unit, f'Representative consumer mismatch: {unit}:{pattern}')
        require(','.join(map(str, representative.get('occurrence', []))) == pattern, f'Representative occurrence mismatch: {unit}:{pattern}')
        require(result.get('occurrence_audit', {}).get('representatives', {}).get(f'{unit}:{pattern}') == representative, f'Aggregate representative mismatch: {unit}:{pattern}')
require(checkpoint_doc.get('chain_head_sha256') == previous, 'Checkpoint chain head mismatch.')
checkpoint_sha = file_sha256(checkpoint)
checkpoint_result = result.get('checkpoint', {})
require(checkpoint_result.get('completed_units') == list(EXPECTED_UNITS), 'Certificate checkpoint completeness mismatch.')
require(checkpoint_result.get('resumed_units') == [], 'Certificate checkpoint contains resumed units.')
require(checkpoint_result.get('fresh_units') == list(EXPECTED_UNITS), 'Certificate checkpoint fresh ledger mismatch.')
require(checkpoint_result.get('chain_head_sha256') == previous, 'Certificate checkpoint chain head mismatch.')
require(checkpoint_result.get('file_sha256') == checkpoint_sha, 'Certificate checkpoint file hash mismatch.')
require(identity.get('checkpoint_structure') == {
    'schema': 'hodge-full-t1-occurrence-checkpoint/v2',
    'unit_order': list(EXPECTED_UNITS),
}, 'Portable identity checkpoint structure mismatch.')
require('checkpoint_integrity' not in identity, 'Portable identity contains raw checkpoint integrity hashes.')
require(scope.get('checkpoint_integrity_external_only') is True, 'Checkpoint integrity scope is not external-only.')
require(scope.get('portable_across_nonstructural_checkpoint_float_changes') is True, 'Portable certificate claim is missing.')
require(scope.get('portable_identity_raw_float_policy') == 'reject every raw float before hashing', 'Portable float policy mismatch.')
require(Path(checkpoint_result.get('path')).resolve() == checkpoint.resolve(), 'Checkpoint path mismatch.')

policy = result.get('nonvacuity_policy', {})
require(policy.get('matrix_indexing') == 'NONVACUITY_MATRIX[moment][ket][bra]', 'Nonvacuity indexing mismatch.')
require(policy.get('matrix') == EXPECTED_NONVACUITY_MATRIX, 'Nonvacuity matrix mismatch.')
unit_results = policy.get('unit_results', {})
require(set(unit_results) == set(EXPECTED_UNITS), 'Nonvacuity result coverage mismatch.')
required_count = 0
for unit in EXPECTED_UNITS:
    moment, route = unit.split(':', 1)
    bra = int(route.split('bra', 1)[1].split('->', 1)[0])
    ket = int(route.rsplit('ket', 1)[1])
    expected_required = EXPECTED_NONVACUITY_MATRIX[moment][ket][bra]
    row = unit_results[unit]
    require(row.get('substantive_required') is expected_required, f'Nonvacuity policy mismatch: {unit}')
    require(row.get('valid') is True, f'Invalid nonvacuity row: {unit}')
    if expected_required:
        required_count += 1
        require(row.get('substantive_observed') is True, f'Required unit is vacuous: {unit}')
        for field in ('matched_h0_support_blocks', 'matched_flux_groups', 'state_pair_tests', 'local_occurrences'):
            require(row.get(field, 0) > 0, f'Required work field is empty: {unit}.{field}')
    else:
        require(bool(row.get('structural_zero_exception')), f'Missing structural-zero justification: {unit}')
require(required_count == 48, f'Unexpected substantive-unit count: {required_count}')

for flag in ('next_stage_authorized', 'gpu_requested', 'gpu_go'):
    require(result.get(flag) is False, f'Forbidden authorization flag: {flag}')
publication = result.get('publication', {})
actual_certificate_bytes = len(certificate.read_bytes())
require(publication.get('finite_serialization_required') is True, 'Finite serialization was not required.')
require(publication.get('post_write_budget_check_required') is True, 'Post-write budget check was not required.')
require(publication.get('serialized_bytes') == actual_certificate_bytes, 'Certificate byte-count mismatch.')
require(actual_certificate_bytes <= publication.get('payload_ceiling_bytes', -1), 'Certificate payload ceiling exceeded.')
require(result.get('operational_report', {}).get('limits', {}).get('max_payload_bytes') == publication.get('payload_ceiling_bytes'), 'Operational payload ceiling mismatch.')
require('cooperative checks' in result.get('wall_clock_enforcement', ''), 'Cooperative wall-clock semantics are missing.')
require('post-write budget check' in result.get('wall_clock_enforcement', ''), 'Post-write budget semantics are missing.')
assert_finite(result)

print(f'FRESH PROMOTABLE PASS: {recomputed_certificate_id}')
print(f'Certificate: {certificate}')
print(f'Checkpoint: {checkpoint}')
print('GPU GO: NO — this artifact stops at the occurrence boundary.')
